In [10]:
!pip install -q kagglehub

In [11]:
import kagglehub
path = kagglehub.dataset_download("thoughtvector/customer-support-on-twitter")
path

Using Colab cache for faster access to the 'customer-support-on-twitter' dataset.


'/kaggle/input/customer-support-on-twitter'

Ccылка на датасет, а также инструкцию по скачке:
https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter

In [12]:
import os
os.listdir(path)

# проверка какие есть файлы папке

['twcs', 'sample.csv']

In [13]:
os.listdir(f'{path}/twcs')

['twcs.csv']

In [14]:
import pandas as pd
data = pd.read_csv(f'{path}/twcs/twcs.csv')
data.shape

(2811774, 7)

In [15]:
data.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [16]:
companies = data[data['inbound'] == False]['author_id'].value_counts()
companies.head(30)

# inbound - это флаг входящее/исходящее обрашение
# author_id - при флаге False это компания от которой исходящее обращение

,count
author_id,
AmazonHelp,169840
AppleSupport,106860
Uber_Support,56270
SpotifyCares,43265
Delta,42253
Tesco,38573
AmericanAir,36764
TMobileHelp,34317
comcastcares,33031


In [17]:
message_company = data[data['inbound'] == False]
message_client = data[data['inbound'] == True]

message_client.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...,"11,13,14",15.0


In [26]:
pairs = message_company.merge(message_client[['tweet_id', 'text']],
                              left_on='in_response_to_tweet_id', right_on='tweet_id',
                              suffixes=('_agent', '_client'))
pairs = pairs[['text_client', 'text_agent']]
pairs.columns = ['input_text', 'target_text']
pairs.head()

,input_text,target_text
0,@sprintcare I have sent several private messag...,@115712 I understand. I would like to assist y...
1,@sprintcare I did.,@115712 Please send us a Private Message so th...
2,@sprintcare is the worst customer service,@115712 Can you please send us a private messa...
3,@sprintcare You gonna magically change your co...,@115713 This is saddening to hear. Please shoo...
4,@sprintcare Since I signed up with you....Sinc...,@115713 We understand your concerns and we'd l...


In [27]:
pairs.shape

(1261888, 2)

In [28]:
import re
def f(a):
    a = re.sub(r'@\w+', '', a)
    a = re.sub(r'http\S+', '', a)
    a = a.strip()
    return a
pairs['input_text']  = pairs['input_text'].apply(f)
pairs['target_text'] = pairs['target_text'].apply(f)
pairs.head()

,input_text,target_text
0,I have sent several private messages and no on...,I understand. I would like to assist you. We w...
1,I did.,Please send us a Private Message so that we ca...
2,is the worst customer service,"Can you please send us a private message, so t..."
3,You gonna magically change your connectivity f...,This is saddening to hear. Please shoot us a D...
4,Since I signed up with you....Since day 1,We understand your concerns and we'd like for ...


In [29]:
pairs = pairs[pairs['input_text'].str.len()  > 0]
pairs = pairs[pairs['target_text'].str.len() > 0]
pairs = pairs[pairs['input_text'].str.len()  >= 5]
pairs = pairs[pairs['target_text'].str.len() >= 5]
pairs = pairs.drop_duplicates()
pairs = pairs.reset_index(drop=True)
pairs.shape #пар после читски

(1241050, 2)

In [30]:
s = pairs.sample(n=1000, random_state=42).reset_index(drop=True)
s.shape

(1000, 2)

In [31]:
train = s[:int(len(s) * 0.8)]
val = s[int(len(s) * 0.8):]
len(train), len(val)

(800, 200)

In [32]:
train.to_csv('train.csv', index=False)
val.to_csv('val.csv',   index=False)
print('succ')

succ
